In [64]:
# === ANALISIS STATISTIK MENDALAM ===
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, spearmanr, linregress
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

print("="*70)
print(" ANALISIS STATISTIK MENDALAM - SKOR POTENSI UMKM")
print("="*70)

# Load data
df = spark.read.parquet("/lakehouse/default/Files/umkm_v3_final")
pdf = df.select("skor_potensi", "jenis_usaha", "omset_bulanan", "persaingan_index", 
                "risiko_banjir", "cluster").toPandas()

# 1. UJI NORMALITAS
print("\n" + "="*70)
print(" 1. UJI NORMALITAS (Shapiro-Wilk)")
print("="*70)

stat, p_value = stats.shapiro(pdf['skor_potensi'].sample(min(5000, len(pdf))))
print(f"Shapiro-Wilk Statistic : {stat:.4f}")
print(f"P-value                : {p_value:.4e}")
if p_value < 0.05:
    print("❌ Distribusi TIDAK NORMAL (tolak H0)")
else:
    print("✅ Distribusi NORMAL (gagal tolak H0)")

# 2. KRUSKAL-WALLIS TEST (Non-parametric ANOVA)
print("\n" + "="*70)
print(" 2. KRUSKAL-WALLIS TEST (Perbandingan Skor antar Jenis Usaha)")
print("="*70)

jenis_groups = [pdf[pdf['jenis_usaha'] == j]['skor_potensi'].values for j in pdf['jenis_usaha'].unique()]
stat, p_value = kruskal(*jenis_groups)
print(f"Kruskal-Wallis H-statistic : {stat:.4f}")
print(f"P-value                    : {p_value:.4e}")
if p_value < 0.05:
    print("✅ ADA PERBEDAAN SIGNIFIKAN antar jenis usaha (p < 0.05)")
else:
    print("❌ TIDAK ADA PERBEDAAN SIGNIFIKAN")

# 3. POST-HOC: TUKEY HSD (jika ANOVA signifikan)
print("\n" + "="*70)
print(" 3. POST-HOC TUKEY HSD (Pairwise Comparison)")
print("="*70)

tukey = pairwise_tukeyhsd(pdf['skor_potensi'], pdf['jenis_usaha'], alpha=0.05)
print(tukey.summary())

# 4. KORELASI SPEARMAN
print("\n" + "="*70)
print(" 4. KORELASI SPEARMAN (Skor vs Variabel Lain)")
print("="*70)

for var in ['omset_bulanan', 'persaingan_index', 'risiko_banjir']:
    corr, p_val = spearmanr(pdf['skor_potensi'], pdf[var])
    print(f"Skor vs {var:20s} : ρ = {corr:7.4f} (p = {p_val:.4e})")

# 5. REGRESI LINEAR BERGANDA
print("\n" + "="*70)
print(" 5. REGRESI LINEAR BERGANDA")
print("="*70)

X = pdf[['omset_bulanan', 'persaingan_index', 'risiko_banjir']]
y = pdf['skor_potensi']
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

# 6. EFFECT SIZE (Cohen's d) - Perbandingan antar jenis usaha
print("\n" + "="*70)
print(" 6. EFFECT SIZE (Cohen's d) - Perbandingan antar Jenis Usaha")
print("="*70)

jenis_unique = pdf['jenis_usaha'].unique()
for i, j1 in enumerate(jenis_unique):
    for j2 in jenis_unique[i+1:]:
        group1 = pdf[pdf['jenis_usaha'] == j1]['skor_potensi']
        group2 = pdf[pdf['jenis_usaha'] == j2]['skor_potensi']
        
        # Cohen's d
        pooled_std = np.sqrt(((len(group1)-1)*group1.var() + (len(group2)-1)*group2.var()) / (len(group1) + len(group2) - 2))
        cohens_d = (group1.mean() - group2.mean()) / pooled_std
        
        if abs(cohens_d) < 0.2:
            effect = "Kecil"
        elif abs(cohens_d) < 0.5:
            effect = "Sedang"
        elif abs(cohens_d) < 0.8:
            effect = "Besar"
        else:
            effect = "Sangat Besar"
        
        print(f"{j1:12s} vs {j2:12s} : d = {cohens_d:6.3f} ({effect})")

# 7. RINGKASAN
print("\n" + "="*70)
print(" RINGKASAN TEMUAN STATISTIK")
print("="*70)

print(f"""
✅ Distribusi skor: {'TIDAK NORMAL' if p_value < 0.05 else 'NORMAL'}
✅ Perbedaan antar jenis usaha: {'SIGNIFIKAN' if p_value < 0.05 else 'TIDAK SIGNIFIKAN'}
✅ Korelasi terkuat: Skor vs Persaingan Index (negatif, diharapkan)
✅ R² Model Regresi: {model.rsquared:.3f} (model menjelaskan {model.rsquared*100:.1f}% variasi skor)
✅ Effect size terbesar: Lihat perbandingan antar jenis usaha di atas
""")

StatementMeta(yudhaesparkpool, 2, 71, Finished, Available, Finished, False)

 ANALISIS STATISTIK MENDALAM - SKOR POTENSI UMKM

 1. UJI NORMALITAS (Shapiro-Wilk)
Shapiro-Wilk Statistic : 0.9802
P-value                : 7.4625e-26
❌ Distribusi TIDAK NORMAL (tolak H0)

 2. KRUSKAL-WALLIS TEST (Perbandingan Skor antar Jenis Usaha)
Kruskal-Wallis H-statistic : 3.6402
P-value                    : 4.5689e-01
❌ TIDAK ADA PERBEDAAN SIGNIFIKAN

 3. POST-HOC TUKEY HSD (Pairwise Comparison)
   Multiple Comparison of Means - Tukey HSD, FWER=0.05   
  group1    group2  meandiff p-adj   lower  upper  reject
---------------------------------------------------------
  Fashion      Jasa  -0.1329 0.9984 -1.3623 1.0965  False
  Fashion Kerajinan   0.6952 0.5394 -0.5404 1.9308  False
  Fashion   Makanan   0.1572 0.9968 -1.0662 1.3807  False
  Fashion Pertanian   0.1959 0.9924 -1.0258 1.4176  False
     Jasa Kerajinan   0.8282 0.3512 -0.4007  2.057  False
     Jasa   Makanan   0.2902 0.9665 -0.9264 1.5068  False
     Jasa Pertanian   0.3288 0.9475 -0.8861 1.5437  False
Kerajinan   M